# Crop Yield Forecast — Simple History Model with Spark ML

**Question:** For a crop already grown in a district, what yield might the next recorded crop year have?

One row is **state × district × crop × season × recorded crop year**. We use only location and earlier yields. We compare two simple rules with Random Forest and gradient boosting, then choose using validation data.

**Existing input:** `data/features/crop_yield_ml_features.parquet`. No upstream notebook needs to run for this version if that file exists.

**Why history only?** Existing seasonal weather has repeated rainfall, inconsistent season dates, and placeholder NDVI phases. Current-year area is also unknown before planting. These columns are excluded here; this notebook does not repair them.

**Scope:** A learning/demo forecast for known crop–district–season combinations. Crop units, source outliers, and reporting dates still need verification. A better score is measured below, never assumed. This notebook does not build an API.

## 1. Setup and settings

Use the project's Python 3.11 environment, PySpark 3.5, and Java 17. Start Jupyter with `JAVA_HOME` pointing to Java 17. The guide gives commands.

Exports are **off** by default. A normal run changes no input data or existing model artifacts. Tables and predictions stay in memory.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import sys

import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql import SparkSession, Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, Imputer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

current = Path.cwd()
project_root = current if (current / "CLAUDE.md").exists() else current.parent.parent
features_file = project_root / "data/features/crop_yield_ml_features.parquet"
if not features_file.is_file():
    raise FileNotFoundError(f"Existing feature file required: {features_file}")

TRAIN_END, VAL_YEAR, TEST_YEAR = 2020, 2021, 2022
EXPORT_ARTIFACTS = False
artifact_dir = project_root / "models/yield_history_v2"
KEY = ["state", "district", "crop_name", "season"]
BASE_COLS = KEY + ["crop_year", "yield", "area", "centroid_lat", "centroid_lon"]

os.environ["PYSPARK_PYTHON"] = sys.executable
spark = (
    SparkSession.builder.appName("Simple Crop Yield Forecast")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version}; input: {features_file.name}; export: {EXPORT_ARTIFACTS}")

Spark 3.5.0; input: crop_yield_ml_features.parquet; export: False


## 2. Load and check the existing records

Keep positive recorded yield and positive area; remove the duplicate-summary season `Total`. This matches the earlier notebook's target scope. Zero yields are excluded pending source review, so this is **not a crop-failure model**. Do not automatically replace suspicious yields with invented values.

Area is used only to reject invalid training records. Neither current-year area nor production is a model input. Yield remains in **source-reported units per hectare**; units are not independently verified, particularly for crops with unusually large values.

In [2]:
raw = spark.read.parquet(str(features_file))
missing = sorted(set(BASE_COLS) - set(raw.columns))
if missing:
    raise ValueError(f"Required columns missing: {missing}")

df = raw.select(*BASE_COLS).withColumn("crop_year", F.col("crop_year").cast("int"))
for column in ["yield", "area", "centroid_lat", "centroid_lon"]:
    df = df.withColumn(column, F.col(column).cast("double"))
for column in KEY:
    df = df.filter(F.col(column).isNotNull() & (F.trim(F.col(column)) != ""))
for column in ["yield", "area"]:
    df = df.filter(
        F.col(column).isNotNull() & ~F.isnan(column)
        & (F.col(column) > 0) & (F.col(column) < float("inf"))
    )
df = df.filter((F.col("season") != "Total") & F.col("crop_year").isNotNull()).cache()

if df.groupBy(*KEY, "crop_year").count().filter("count > 1").limit(1).count():
    raise ValueError("Duplicate crop-year keys: resolve them before training.")
for column, bound in [("centroid_lat", 90), ("centroid_lon", 180)]:
    if df.filter(F.isnan(column) | (F.abs(F.col(column)) > bound)).limit(1).count():
        raise ValueError(f"Invalid coordinates in {column}.")

last_year = df.agg(F.max("crop_year")).first()[0]
if last_year is None:
    raise ValueError("No usable records remain.")
TARGET_YEAR = int(last_year) + 1
assert TRAIN_END < VAL_YEAR < TEST_YEAR <= last_year
print(f"Source rows: {raw.count():,}; usable positive-yield rows: {df.count():,}")
print(f"Latest recorded year: {last_year}; one-step forecast year: {TARGET_YEAR}")
df.groupBy("crop_year").count().orderBy("crop_year").show()
print("Largest source yields below are audit candidates, not verified mistakes:")
df.orderBy(F.desc("yield")).select(*KEY, "crop_year", "yield", "area").show(5, truncate=False)

Source rows: 93,262; usable positive-yield rows: 81,774
Latest recorded year: 2023; one-step forecast year: 2024
+---------+-----+
|crop_year|count|
+---------+-----+
|     2016|10690|
|     2017|11044|
|     2018|11529|
|     2019|11236|
|     2020|10934|
|     2021|11021|
|     2022|10522|
|     2023| 4798|
+---------+-----+

Largest source yields below are audit candidates, not verified mistakes:
+--------------+-------------+---------+----------+---------+---------+------+
|state         |district     |crop_name|season    |crop_year|yield    |area  |
+--------------+-------------+---------+----------+---------+---------+------+
|Assam         |Tinsukia     |Coconut  |Whole Year|2018     |43958.333|72.0  |
|Assam         |Tinsukia     |Coconut  |Whole Year|2021     |37469.136|81.0  |
|Assam         |Tinsukia     |Coconut  |Whole Year|2020     |37469.136|81.0  |
|Assam         |Tinsukia     |Coconut  |Whole Year|2022     |37468.354|79.0  |
|Andhra Pradesh|West Godavari|Coconut  |Whol

## 3. Build a small set of past-only features

| Input | Plain meaning |
|---|---|
| `crop_name`, `season`, `state` | Which crop, season, and state? |
| `centroid_lat`, `centroid_lon` | Where is the district? |
| `yield_lag1` | Yield in exactly the previous recorded year; missing if that year is absent |
| `yield_roll3` | Mean yield from the previous three recorded years, excluding the target year |
| `history_count` | Number of usable yields in that three-year window, from 0 to 3 |
| `history_age` | Years since the latest yield in that window; missing if no history exists |
| `has_lag1` | Whether the previous-year yield exists |

The same feature function builds training rows and forecast rows. Missing numbers are filled from training medians inside the model pipeline. Counts and flags preserve the fact that history was missing.

In [3]:
CAT_COLS = ["crop_name", "season", "state"]
NUM_COLS = [
    "centroid_lat", "centroid_lon", "yield_lag1", "yield_roll3",
    "history_count", "history_age", "has_lag1",
]


def add_history_features(rows):
    """Use exact previous-year yield and only the preceding three years."""
    window = Window.partitionBy(*KEY).orderBy("crop_year")
    previous = window.rangeBetween(-3, -1)
    return (
        rows.withColumn("yield_lag1", F.max("yield").over(window.rangeBetween(-1, -1)))
        .withColumn("yield_roll3", F.avg("yield").over(previous))
        .withColumn("history_count", F.count("yield").over(previous).cast("double"))
        .withColumn("latest_history_year", F.max("crop_year").over(previous))
        .withColumn("history_age", (F.col("crop_year") - F.col("latest_history_year")).cast("double"))
        .withColumn("has_lag1", F.col("yield_lag1").isNotNull().cast("double"))
        .withColumn("log_yield", F.log1p("yield"))
    )


model_df = add_history_features(df).cache()
assert model_df.filter(F.col("latest_history_year") >= F.col("crop_year")).count() == 0
assert model_df.filter(~F.col("history_count").between(0, 3)).count() == 0
model_df.orderBy(*KEY, "crop_year").select(
    *KEY, "crop_year", "yield", "yield_lag1", "yield_roll3", "history_count", "history_age"
).show(8, truncate=False)

+--------------+--------+---------+------+---------+-----+----------+-------------------+-------------+-----------+
|state         |district|crop_name|season|crop_year|yield|yield_lag1|yield_roll3        |history_count|history_age|
+--------------+--------+---------+------+---------+-----+----------+-------------------+-------------+-----------+
|Andhra Pradesh|Chittoor|Arhar/Tur|Kharif|2016     |0.113|NULL      |NULL               |0.0          |NULL       |
|Andhra Pradesh|Chittoor|Arhar/Tur|Kharif|2017     |0.254|0.113     |0.113              |1.0          |1.0        |
|Andhra Pradesh|Chittoor|Arhar/Tur|Kharif|2018     |0.162|0.254     |0.1835             |2.0          |1.0        |
|Andhra Pradesh|Chittoor|Arhar/Tur|Kharif|2019     |0.077|0.162     |0.17633333333333334|3.0          |1.0        |
|Andhra Pradesh|Chittoor|Arhar/Tur|Kharif|2020     |0.123|0.077     |0.16433333333333336|3.0          |1.0        |
|Andhra Pradesh|Chittoor|Arhar/Tur|Kharif|2021     |0.115|0.123     |0.1

## 4. Split by year

Train through **2020**, choose the method on **2021**, and report its result on **2022** after refitting through 2021. Partial 2023 records are used only for the final forecast fit.

The split tests later recorded years for the represented districts. It does not test new districts, independently verify upstream year labels, or prove that official prior-year reports were already published before planting.

In [4]:
train = model_df.filter(F.col("crop_year") <= TRAIN_END)
val = model_df.filter(F.col("crop_year") == VAL_YEAR)
train_val = model_df.filter(F.col("crop_year") <= VAL_YEAR)
test = model_df.filter(F.col("crop_year") == TEST_YEAR)
for name, rows in [("train", train), ("validation", val), ("test", test)]:
    count = rows.count()
    if count == 0:
        raise ValueError(f"No {name} rows for the configured year split.")
    print(f"{name}: {count:,} rows")

train: 55,433 rows
validation: 11,021 rows
test: 10,522 rows


## 5. Compare two simple rules and two tree models

- **Last year:** previous-year yield, then recent three-year mean, then training crop mean.
- **Recent mean:** recent three-year mean, then training crop mean.
- **Random Forest:** many trees averaged together.
- **GBT:** trees that successively correct earlier prediction errors.

All four methods compete on the same validation rows. Lowest **log-RMSE** wins, including a simple rule if it wins. Log-RMSE is not a percentage error and does not give equal weight to each crop. Per-crop results below expose differences.

Tree models encode categories, fill missing numeric values with training medians, and predict `log1p(yield)`. `expm1` returns the source yield scale. This transform does not correct source units or outliers.

In [5]:
BASELINES = ["last_year", "recent_mean"]
CANDIDATES = {
    "random_forest": lambda: RandomForestRegressor(
        numTrees=100, maxDepth=10, maxBins=128, seed=42
    ),
    "gbt": lambda: GBTRegressor(
        maxIter=80, maxDepth=4, stepSize=0.1, maxBins=128, seed=42
    ),
}


def make_pipeline(estimator):
    """Fit category encoding, numeric medians, and the regressor together."""
    indexed = [column + "_idx" for column in CAT_COLS]
    imputed = [column + "_imp" for column in NUM_COLS]
    return Pipeline(stages=[
        StringIndexer(inputCols=CAT_COLS, outputCols=indexed, handleInvalid="keep"),
        Imputer(inputCols=NUM_COLS, outputCols=imputed, strategy="median"),
        VectorAssembler(inputCols=indexed + imputed, outputCol="features"),
        estimator.setFeaturesCol("features").setLabelCol("log_yield"),
    ])


def fit_method(name, history):
    """Fit one simple rule or one Spark ML pipeline on the supplied years."""
    if name in BASELINES:
        averages = history.groupBy("crop_name").agg(F.avg("yield").alias("crop_average"))
        return {"name": name, "crop_average": averages, "pipeline": None}
    return {"name": name, "crop_average": None,
            "pipeline": make_pipeline(CANDIDATES[name]()).fit(history)}


def predict_rows(fitted, rows):
    """Return log-scale prediction and non-negative source-scale yield."""
    if fitted["name"] in BASELINES:
        predictions = rows.join(fitted["crop_average"], "crop_name", "left")
        past = ["yield_lag1", "yield_roll3"] if fitted["name"] == "last_year" else ["yield_roll3"]
        predictions = predictions.withColumn(
            "prediction", F.log1p(F.coalesce(*past, "crop_average"))
        )
    else:
        predictions = fitted["pipeline"].transform(rows)
    return predictions.withColumn("pred_yield", F.greatest(F.expm1("prediction"), F.lit(0.0)))


def score(predictions):
    """Report log-RMSE and source-scale errors on the same labeled rows."""
    if predictions.filter(
        F.col("prediction").isNull() | F.isnan("prediction")
        | (F.abs(F.col("prediction")) == float("inf"))
    ).limit(1).count():
        raise ValueError("Cannot score every row; check missing crop history and model outputs.")
    metrics = {}
    for metric, label, predicted in [
        ("rmse_log", "log_yield", "prediction"),
        ("rmse", "yield", "pred_yield"),
        ("mae", "yield", "pred_yield"),
        ("r2", "yield", "pred_yield"),
    ]:
        evaluator = "rmse" if metric == "rmse_log" else metric
        metrics[metric] = RegressionEvaluator(
            labelCol=label, predictionCol=predicted, metricName=evaluator
        ).evaluate(predictions)
    return metrics

In [6]:
validation_rows = []
for name in BASELINES + list(CANDIDATES):
    fitted = fit_method(name, train)
    predictions = predict_rows(fitted, val).cache()
    validation_rows.append({"method": name, **score(predictions)})
    predictions.unpersist()
    print(f"{name}: validation log-RMSE = {validation_rows[-1]['rmse_log']:.4f}")

validation_results = pd.DataFrame(validation_rows).sort_values(["rmse_log", "method"])
best_name = validation_results.iloc[0]["method"]
print(validation_results.round(4).to_string(index=False))
print("Chosen using validation only:", best_name)

last_year: validation log-RMSE = 0.3616
recent_mean: validation log-RMSE = 0.2704
random_forest: validation log-RMSE = 0.2723
gbt: validation log-RMSE = 0.2846
       method  rmse_log     rmse     mae     r2
  recent_mean    0.2704 287.1013 20.9149 0.9099
random_forest    0.2723 404.0429 28.4697 0.8216
          gbt    0.2846 431.3757 30.9913 0.7967
    last_year    0.3616 380.0346 22.3804 0.8422
Chosen using validation only: recent_mean


## 6. Evaluate on 2022 and inspect each crop

Refit the chosen method and the last-year rule on the same 2016–2021 records. Do not change the choice using this test table. Since this project already inspected 2022 during earlier work, this is a development holdout rather than a new independent benchmark.

MAE describes typical absolute error for a crop. R² can be negative; that indicates poor predictions relative to a constant fitted to the test outcomes. Undefined R² is left missing. Raw errors pooled across crops need care because source scales differ.

In [7]:
evaluation_methods = list(dict.fromkeys(["last_year", best_name]))
test_predictions = {}
test_rows = []
for name in evaluation_methods:
    fitted = fit_method(name, train_val)
    predictions = predict_rows(fitted, test).cache()
    test_predictions[name] = predictions
    test_rows.append({"method": name, **score(predictions)})

test_results = pd.DataFrame(test_rows)
print(test_results.round(4).to_string(index=False))
base_error = test_results.loc[test_results.method == "last_year", "rmse_log"].iloc[0]
chosen_error = test_results.loc[test_results.method == best_name, "rmse_log"].iloc[0]
print(f"Chosen versus last-year rule: {100 * (1 - chosen_error / base_error):.1f}% lower test log-RMSE")

     method  rmse_log     rmse     mae     r2
  last_year    0.2913 375.5985 26.2032 0.8869
recent_mean    0.2459 355.7570 24.7357 0.8985
Chosen versus last-year rule: 15.6% lower test log-RMSE


In [8]:
def per_crop_metrics(predictions):
    """Keep every crop; compare each crop in its own source yield units."""
    squared_error = (F.col("yield") - F.col("pred_yield")) ** 2
    log_error = (F.col("log_yield") - F.col("prediction")) ** 2
    summary = predictions.groupBy("crop_name").agg(
        F.count("*").alias("n"),
        F.avg("yield").alias("mean_yield"),
        F.sqrt(F.avg(squared_error)).alias("rmse"),
        F.avg(F.abs(F.col("yield") - F.col("pred_yield"))).alias("mae"),
        F.sqrt(F.avg(log_error)).alias("rmse_log"),
        F.sum(squared_error).alias("sse"),
        (F.var_pop("yield") * F.count("*")).alias("sst"),
    )
    return summary.withColumn(
        "r2", F.when(F.col("sst") > 0, 1 - F.col("sse") / F.col("sst"))
    ).drop("sse", "sst")


baseline_by_crop = per_crop_metrics(test_predictions["last_year"]).select(
    "crop_name", F.col("rmse_log").alias("baseline_rmse_log")
)
crop_report = (
    per_crop_metrics(test_predictions[best_name]).join(baseline_by_crop, "crop_name")
    .withColumn("beats_baseline", F.col("rmse_log") < F.col("baseline_rmse_log"))
    .orderBy(F.desc("n"))
)
crop_report_pd = crop_report.toPandas()
print("Top 10 by test rows; the full crop_report retains every crop:")
print(crop_report_pd.head(10).round(4).to_string(index=False))
print("Crops with lower test log-RMSE than last year:",
      int(crop_report_pd.beats_baseline.sum()), "of", len(crop_report_pd))
print("Average per-crop log-RMSE:", round(float(crop_report_pd.rmse_log.mean()), 4))
print("Largest absolute test errors for source review:")
test_predictions[best_name].withColumn(
    "absolute_error", F.abs(F.col("yield") - F.col("pred_yield"))
).orderBy(F.desc("absolute_error")).select(
    *KEY, "crop_year", "yield", "pred_yield", "absolute_error"
).show(5, truncate=False)

Top 10 by test rows; the full crop_report retains every crop:
        crop_name   n  mean_yield    rmse    mae  rmse_log      r2  baseline_rmse_log  beats_baseline
            Maize 675      3.2748  0.7490 0.5124    0.1898  0.8141             0.1935            True
             Rice 641      2.8071  2.2868 0.5269    0.2228  0.0740             0.2206           False
Moong(Green Gram) 505      0.6347  0.1988 0.1365    0.1214  0.4926             0.1350            True
             Urad 478      0.6906  0.1924 0.1289    0.1124  0.4455             0.1247            True
          Sesamum 407      0.5087  0.1583 0.1036    0.1003  0.6188             0.1119            True
        Groundnut 368      3.4616 31.1404 1.8810    0.3079 -0.0026             0.3150            True
            Wheat 348      2.9343  0.4116 0.2830    0.1115  0.8622             0.1062           False
             Gram 340      1.2421  0.2702 0.1759    0.1126  0.6178             0.1174            True
            Onion 33

## 7. Fit once on available history and forecast the next year

With the supplied file, latest recorded year is 2023 and the forecast is **2024**, not today's year. We forecast only combinations observed in the last three years. Missing 2023 stays missing; a 2022 observation is never relabeled as last year's yield.

Only the next recorded year is supported. A request for a later year needs newer observed yields or a separately evaluated multi-year method. We do not feed predictions back as observed history.

Forecasts assume prior recorded-year reports are available. Source publication dates are absent, so a real pre-planting release needs that availability check.

In [9]:
def build_forecast_rows(history, target_year):
    """Build one-step rows using the same history feature code as training."""
    latest_year = history.agg(F.max("crop_year")).first()[0]
    if latest_year is None or target_year != latest_year + 1:
        raise ValueError("Only the next year after the supplied observed history is supported.")
    recent = history.filter(F.col("crop_year").between(target_year - 3, target_year - 1))
    latest_row = Window.partitionBy(*KEY).orderBy(F.desc("crop_year"))
    future = (
        recent.withColumn("row_number", F.row_number().over(latest_row))
        .filter(F.col("row_number") == 1)
        .select(*KEY, F.lit(target_year).cast("int").alias("crop_year"),
                F.lit(None).cast("double").alias("yield"),
                F.lit(None).cast("double").alias("area"),
                "centroid_lat", "centroid_lon")
    )
    combined = history.select(*BASE_COLS).unionByName(future)
    return add_history_features(combined).filter(F.col("crop_year") == target_year)


final_fit = fit_method(best_name, model_df)
forecast_rows = build_forecast_rows(df, TARGET_YEAR).cache()
forecast_predictions = (
    predict_rows(final_fit, forecast_rows)
    .withColumn("change_vs_recent_pct", 100 * (F.col("pred_yield") / F.col("yield_roll3") - 1))
    .withColumn("history_status", F.when(F.col("has_lag1") == 1, "previous_year_available")
                .otherwise("previous_year_missing"))
    .withColumn("method", F.lit(best_name))
    .withColumn("yield_unit", F.lit("source yield per hectare; crop units unverified"))
    .select(*KEY, "crop_year", "pred_yield", "yield_lag1", "yield_roll3",
            "history_count", "history_age", "history_status", "change_vs_recent_pct",
            "method", "yield_unit")
    .cache()
)
print(f"Forecast rows for {TARGET_YEAR}: {forecast_predictions.count():,}")
forecast_predictions.groupBy("history_status").count().show()

Forecast rows for 2024: 11,957
+--------------------+-----+
|      history_status|count|
+--------------------+-----+
|previous_year_ava...| 4798|
|previous_year_mis...| 7159|
+--------------------+-----+



## 8. Ask questions using the forecasts

| Question | What this version answers |
|---|---|
| What yield might this district's crop have next recorded year? | A point estimate for a recently observed combination |
| How does that compare with its recent history? | Forecast versus the available previous-three-year mean |
| How do districts compare for the same crop and season? | Compare forecast rows on the same source scale |
| Which local crops look above their own state averages? | A relative-performance shortlist among recently grown crops |
| How much past data supports the forecast? | History count, age, and previous-year availability; these are not confidence probabilities |
| How well did the method work for this crop? | The 2022 per-crop report against the last-year rule |

**Not answered:** rain/NDVI what-if effects, causes of yield change, crop-failure probability, prediction intervals, profit, a newly introduced crop's suitability, a farm-specific forecast, or arbitrary future years.

Crop comparisons use predicted yield divided by the **unweighted mean of reported historical district-year yields** for the same state, crop, and season. This is not area-weighted state productivity or a profit ranking. Ratios are ranked before rounding.

In [10]:
state_reference = df.groupBy("state", "crop_name", "season").agg(
    F.avg("yield").alias("state_avg_yield")
)
rank_window = Window.partitionBy("state", "district", "season").orderBy(
    F.desc("relative_score"), "crop_name"
)
crop_comparisons = (
    forecast_predictions.join(state_reference, ["state", "crop_name", "season"])
    .withColumn("relative_score", F.col("pred_yield") / F.col("state_avg_yield"))
    .withColumn("rank", F.row_number().over(rank_window))
)


def predict_yield(state, district, crop, season, target_year=TARGET_YEAR):
    """Look up a supported next-year forecast, with its history context."""
    if target_year != TARGET_YEAR:
        raise ValueError(f"This forecast snapshot supports only {TARGET_YEAR}.")
    row = forecast_predictions.filter(
        (F.col("state") == state) & (F.col("district") == district)
        & (F.col("crop_name") == crop) & (F.col("season") == season)
    ).first()
    return row.asDict() if row is not None else None


def compare_crops(state, district, season, top_n=5):
    """Return relative yield comparisons, not planting or profit advice."""
    if isinstance(top_n, bool) or not isinstance(top_n, int) or top_n < 1:
        raise ValueError("top_n must be a positive integer.")
    return (
        crop_comparisons.filter(
            (F.col("state") == state) & (F.col("district") == district)
            & (F.col("season") == season)
        ).orderBy("rank").limit(top_n)
        .select("rank", "crop_name", "pred_yield", "state_avg_yield", "relative_score",
                "history_count", "history_age", "history_status")
        .toPandas()
    )


example = forecast_predictions.filter(F.col("crop_name") == "Wheat").orderBy(*KEY).first()
if example is None:
    example = forecast_predictions.orderBy(*KEY).first()
print(predict_yield(example.state, example.district, example.crop_name, example.season))
print(compare_crops(example.state, example.district, example.season).round(3).to_string(index=False))

{'state': 'Andhra Pradesh', 'district': 'Guntur', 'crop_name': 'Wheat', 'season': 'Rabi', 'crop_year': 2024, 'pred_yield': 1.3335, 'yield_lag1': None, 'yield_roll3': 1.3335, 'history_count': 2.0, 'history_age': 2.0, 'history_status': 'previous_year_missing', 'change_vs_recent_pct': 0.0, 'method': 'recent_mean', 'yield_unit': 'source yield per hectare; crop units unverified'}
 rank         crop_name  pred_yield  state_avg_yield  relative_score  history_count  history_age          history_status
    1             Jowar       5.861            3.093           1.895            3.0          1.0 previous_year_available
    2 Moong(Green Gram)       1.272            0.803           1.584            3.0          1.0 previous_year_available
    3     Small Millets       1.575            1.012           1.556            2.0          2.0   previous_year_missing
    4            Potato      23.667           15.501           1.527            1.0          2.0   previous_year_missing
    5      Dry Ch

## 9. Optional export for a later API

Leave `EXPORT_ARTIFACTS = False` for review. When enabled, export the chosen pipeline (or the baseline's crop averages), forecast features, predictions, crop comparisons, all per-crop results, and metadata into one new `models/yield_history_v2/` folder. An existing folder raises an error; choose a new version name for another export.

An API can read saved predictions and comparisons with pandas/pyarrow. Those batch lookups do not require Spark. Recomputing predictions with a saved Spark pipeline does require Spark and the same feature-building rules. No HTTP routes are created here.

In [11]:
metadata = {
    "version": "history_v2",
    "method": best_name,
    "target": "log1p(yield); expm1(prediction) returns source yield per hectare",
    "yield_unit": "source yield per hectare; crop units unverified",
    "source": str(features_file.relative_to(project_root)),
    "source_sha256": hashlib.sha256(features_file.read_bytes()).hexdigest(),
    "categorical_features": CAT_COLS,
    "numeric_features": NUM_COLS,
    "history_window_years": 3,
    "trained_through": int(last_year),
    "forecast_year": TARGET_YEAR,
    "train_end": TRAIN_END,
    "validation_year": VAL_YEAR,
    "test_year": TEST_YEAR,
    "validation_results": validation_results.to_dict(orient="records"),
    "test_results": test_results.to_dict(orient="records"),
    "limits": ["positive recorded yields only", "units and outliers not independently verified",
               "recorded year labels and report availability need source checks",
               "no weather what-if, farm-level, profit, or multi-year inference"],
}

if EXPORT_ARTIFACTS:
    if artifact_dir.exists():
        raise FileExistsError(f"Choose a new artifact_dir; already exists: {artifact_dir}")
    if final_fit["pipeline"] is not None:
        final_fit["pipeline"].write().save(str(artifact_dir / "pipeline"))
    else:
        final_fit["crop_average"].write.parquet(str(artifact_dir / "crop_averages"))
    forecast_rows.write.parquet(str(artifact_dir / "forecast_features"))
    forecast_predictions.write.parquet(str(artifact_dir / "yield_predictions"))
    crop_comparisons.write.parquet(str(artifact_dir / "crop_comparisons"))
    crop_report.write.parquet(str(artifact_dir / "per_crop_test_metrics"))
    (artifact_dir / "metadata.json").write_text(json.dumps(metadata, indent=2))
    print("Exported:", artifact_dir)
else:
    print("Export disabled; existing data and model artifacts were not changed.")

Export disabled; existing data and model artifacts were not changed.


## 10. What this run found

**The recent three-year mean won validation.** It scored 0.2704 log-RMSE, versus 0.2723 for Random Forest and 0.2846 for GBT. The simplest winner is the selected forecast method; this run does not export a trained tree as its winner.

On the 2022 development holdout, recent-mean log-RMSE was **0.2459**, versus **0.2913** for the last-year rule: **15.6% lower**. It improved log-RMSE for **41 of 56 crops**. Rice, Wheat, and Onion did not improve by that metric. Groundnut still has poor R² and needs source review. The full per-crop report matters more than one overall claim.

For supported forecast rows, the chosen rule equals `yield_roll3`, so `change_vs_recent_pct` is zero by construction. That is a baseline expectation, not evidence that future yield will be unchanged. The prepared snapshot contains **11,957 forecasts for 2024**; **7,159** lack a 2023 yield.

A useful first version must beat a simple rule for the crops the team cares about and expose its limits. Existing data supports this clear history-based demonstration. Stronger claims need verified units/outliers, crop-year and report dates, and fresh evaluation years. Add corrected rainfall, NDVI, irrigation, or temperature only as separately evaluated next steps.

Exports were disabled for this run. The upstream notebooks, input datasets, and existing saved models were not changed. See the guide for all measured results and the supported inference questions.

In [12]:
for cached_rows in [df, model_df, forecast_rows, forecast_predictions, *test_predictions.values()]:
    cached_rows.unpersist()
# Keep Spark available for the inference examples; call spark.stop() when finished.